# DataAtlas baseline on iNaturalist (five SURF models)

This notebook runs the published DataAtlas metadata and description pipeline on each of the 100 iNaturalist species CSV files separately. Its prompts and 15-word output constraint are preserved; only the LLM transport is redirected to SURF/Willma.

Restart behavior is persistent:

- `{species}_dataatlas.json` means the dataset succeeded.
- `{species}_dataatlas.skipped.json` means it reached the 10-minute timeout.
- Both are skipped on the next run. Other failures are logged and retried.
- Once all 100 datasets have either result, the whole model is skipped and execution continues with the next model.

After a kernel or machine restart, use **Run All**. You can also rerun only the final cell if the setup cell has already run in the current kernel.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

BASE = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/config.py').is_file()),
    None,
)
if BASE is None:
    raise FileNotFoundError('Could not locate the metadata_agent project root')
sys.path.insert(0, str(BASE))
load_dotenv(BASE / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

# The order used by the M³ and AutoDDG comparisons.
MODELS = [
    'default-text-large',
    'Qwen/Qwen3.6-35B-A3B-FP8',
    'Sehyo/Qwen3.5-122B-A10B-NVFP4',
    'mistralai/Devstral-Small-2-24B-Instruct-2512',
    'mistralai/Mistral-Small-3.2-24B-Instruct-2506',
]
INATURALIST_DIR_CANDIDATES = [
    BASE / 'data/experiment/inaturalist',
    BASE / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
]
OUTPUTS_ROOT = BASE / 'outputs'
DOMAIN = 'Biodiversity and species occurrence'
PROCESS_TIMEOUT_SECONDS = 600
MAX_ROWS = 20_000_000
EXPECTED_CSV_COUNT = 100


def resolve_inaturalist_dir() -> Path:
    for path in INATURALIST_DIR_CANDIDATES:
        if path.is_dir() and any(path.glob('*.csv')):
            return path
    searched = '\n  '.join(str(path) for path in INATURALIST_DIR_CANDIDATES)
    raise FileNotFoundError(f'iNaturalist data directory not found. Tried:\n  {searched}')


if not os.getenv('SURF_API_KEY'):
    raise ValueError('SURF_API_KEY is missing from metadata_agent/.env')
if not os.getenv('SURF_API_BASE'):
    raise ValueError('SURF_API_BASE is missing from metadata_agent/.env')

try:
    import data_dictionary_generator
except ImportError as exc:
    raise ImportError(
        'Install DataAtlas first with:\n'
        'python -m pip install -e '
        '/home/com3dian/Github/process_data_metadata_agent/'
        'data-dictionary-generator --no-deps'
    ) from exc

from src.baseline.dataatlas.batch_runner import run_all_models

DATA_DIR = resolve_inaturalist_dir()
csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f'Project: {BASE}')
print(f'DataAtlas: v{data_dictionary_generator.__version__} from {data_dictionary_generator.__file__}')
print(f'iNaturalist directory: {DATA_DIR}')
print(f'iNaturalist files: {len(csv_files)}')
print(f'Models: {len(MODELS)}')
if len(csv_files) != EXPECTED_CSV_COUNT:
    raise ValueError(f'Expected {EXPECTED_CSV_COUNT} iNaturalist CSV files, found {len(csv_files)}')

## Run or resume all five models

This cell proceeds through all models even when an individual dataset fails. Restarting it skips valid successes and persistent timeout records.

In [ ]:
summaries = run_all_models(
    data_dir=DATA_DIR,
    outputs_root=OUTPUTS_ROOT,
    models=MODELS,
    timeout_seconds=PROCESS_TIMEOUT_SECONDS,
    domain=DOMAIN,
    max_rows=MAX_ROWS,
    experiment='inat',
    dataset_label='iNaturalist',
)

import pandas as pd
pd.DataFrame(summaries)